In [123]:
import pandas as pd

In [124]:
import numpy as np


In [125]:
df = pd.read_csv(r"C:\Users\bhard\Desktop\GRIDATHON\Astram event data_anonymized - Astram event data_anonymizedb40ac87 (2).csv")

In [126]:
pip install deep-translator scikit-learn xgboost joblib langdetect

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [127]:
import pandas as pd 
import numpy as np 
import re 
from deep_translator import GoogleTranslator
from langdetect import detect


In [128]:
# Parse all datetime columns first
dt_cols = ['start_datetime', 'created_date', 'end_datetime',
           'resolved_datetime', 'closed_datetime', 'modified_datetime']

for col in dt_cols:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')


In [129]:
df['effective_end_dt'] = (
    df['closed_datetime']
    .fillna(df['resolved_datetime'])   # if closed is NaN, try resolved
    .fillna(df['end_datetime'])        # if still NaN, try end
    .fillna(df['modified_datetime'])   # last resort
)

In [130]:

df['effective_start_dt'] = (
    df['start_datetime']
    .fillna(df['created_date'])        # if start is NaN, fall back to created
)

In [131]:
# Now compute duration cleanly
df['duration_min'] = (
    df['effective_end_dt'] - df['effective_start_dt']
).dt.total_seconds() / 60


In [132]:
# Step 1: drop the 2 missing
df = df.dropna(subset=['duration_min'])

# Step 2: remove negatives (impossible — data errors)
df = df[df['duration_min'] > 0]

# Step 3: cap outliers using 99th percentile (domain-safe)
p99 = df['duration_min'].quantile(0.99)
print(f"99th percentile: {p99:.1f} min ({p99/60:.1f} hrs)")

df = df[df['duration_min'] <= p99]

print(f"\nClean rows: {len(df)}")
print(df['duration_min'].describe().round(1))

99th percentile: 79692.8 min (1328.2 hrs)

Clean rows: 8033
count     8033.0
mean      1839.5
std       7826.9
min          0.0
25%         32.7
50%        126.8
75%        165.4
max      79376.6
Name: duration_min, dtype: float64


In [133]:
# Stage 1: classify incident type
df['is_extended'] = (df['duration_min'] > 720).astype(int)
# 0 = normal incident (≤12 hrs)
# 1 = extended incident (>12 hrs) → needs sustained deployment

# Stage 2a: predict duration for normal incidents
df_normal   = df[df['is_extended'] == 0]   # ~7800 rows

# Stage 2b: predict resource needs for extended incidents separately
df_extended = df[df['is_extended'] == 1]   # the outlier group


In [134]:
STRUCT_FEATURES = [
    'hour', 'day_of_week', 'month', 'is_night',
    'reporting_delay_min',
    'latitude', 'longitude',
    'requires_road_closure',
    'event_cause_enc', 'veh_type_enc', 'corridor_enc',
    'police_station_enc', 'zone_enc'
]

In [135]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import re

# ── Temporal features ─────────────────────────────────────────
df['hour']        = df['effective_start_dt'].dt.hour
df['day_of_week'] = df['effective_start_dt'].dt.dayofweek
df['month']       = df['effective_start_dt'].dt.month
df['is_night']    = df['hour'].apply(lambda h: 1 if (h >= 21 or h <= 6) else 0)
df['reporting_delay_min'] = (
    df['created_date'] - df['effective_start_dt']
).dt.total_seconds() / 60
df['reporting_delay_min'] = df['reporting_delay_min'].fillna(0)

# ── Encode categoricals (initial pass) ───────────────────────
cat_cols = ['event_cause', 'veh_type', 'corridor', 'police_station', 'zone']
le = {}
for col in cat_cols:
    df[col] = df[col].fillna('unknown')
    le[col] = LabelEncoder()
    df[col + '_enc'] = le[col].fit_transform(df[col])

df['requires_road_closure'] = df['requires_road_closure'].astype(int)

# ── Translate Kannada → English (needed for NLP extraction below) ──
def translate_to_english(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text)
    text = re.sub(r'\[LOCATION\]', 'location', text)
    text = re.sub(r'\[PERSON\]', 'person', text)
    try:
        from langdetect import detect
        lang = detect(text)
        if lang == 'kn':
            from deep_translator import GoogleTranslator
            return GoogleTranslator(source='kn', target='en').translate(text)
        return text
    except:
        return text

print("Translating descriptions for NLP keyword extraction...")
df['desc_en'] = df['description'].apply(translate_to_english)
print(f"Done. df shape: {df.shape}")

Translating descriptions for NLP keyword extraction...
Done. df shape: (8033, 61)


In [136]:
# ═══════════════════════════════════════════════════════════════
# CELL: NLP Extraction — fill missing structured cols from description
# ═══════════════════════════════════════════════════════════════

EVENT_CAUSE_KEYWORDS = {
    'vehicle_breakdown': ['breakdown', 'starting problem', 'engine', 'tyre', 'puncture',
                          'fuel', 'battery', 'stalled', 'stopped', 'pipe', 'vehicle off',
                          'lcv', 'bus', 'truck', 'lorry', 'tanker'],
    'tree_fall':         ['tree', 'tree fall', 'fallen tree', 'branch'],
    'accident':          ['accident', 'collision', 'crash', 'hit', 'injured', 'injury'],
    'flooding':          ['flood', 'water', 'waterlog', 'rain', 'drain', 'chamber', 'sewage'],
    'road_damage':       ['pothole', 'road damage', 'crater', 'cave', 'manhole'],
}

VEH_TYPE_KEYWORDS = {
    'heavy_vehicle': ['truck', 'lorry', 'heavy', 'tanker', 'trailer', 'container'],
    'lcv':           ['lcv', 'light commercial', 'tempo', 'mini truck', 'pickup'],
    'private_bus':   ['bus', 'private bus', 'school bus', 'volvo'],
    'two_wheeler':   ['bike', 'motorcycle', 'scooter', 'two wheeler', 'motorbike'],
    'car':           ['car', 'sedan', 'suv', 'jeep', 'cab', 'taxi'],
    'auto':          ['auto', 'autorickshaw', 'three wheeler'],
}

PRIORITY_HIGH_KEYWORDS = ['accident', 'injury', 'injured', 'fire', 'flood',
                           'road closure', 'blocked', 'tree fall', 'major']

def extract_event_cause(text):
    if pd.isna(text): return None
    text = text.lower()
    for cause, keywords in EVENT_CAUSE_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return cause
    return None

def extract_veh_type(text):
    if pd.isna(text): return None
    text = text.lower()
    for vtype, keywords in VEH_TYPE_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return vtype
    return None

def extract_priority(text):
    if pd.isna(text): return None
    text = text.lower()
    return 'High' if any(kw in text for kw in PRIORITY_HIGH_KEYWORDS) else 'Low'

# Apply extraction only to fill NaN values (never overwrite existing data)
df['event_cause_nlp'] = df['desc_en'].apply(extract_event_cause)
df['veh_type_nlp']    = df['desc_en'].apply(extract_veh_type)
df['priority_nlp']    = df['desc_en'].apply(extract_priority)

before_cause = df['event_cause'].isna().sum() if df['event_cause'].dtype == object else (df['event_cause'] == 'unknown').sum()
df['event_cause'] = df['event_cause'].replace('unknown', None).combine_first(df['event_cause_nlp']).fillna('others')
df['veh_type']    = df['veh_type'].replace('unknown', None).combine_first(df['veh_type_nlp']).fillna('unknown')
df['priority']    = df['priority'].replace('unknown', None).combine_first(df['priority_nlp']).fillna('Low')

print(f"event_cause filled from NLP: {(df['event_cause_nlp'].notna()).sum()} rows")
print(f"\nevent_cause distribution:\n{df['event_cause'].value_counts()}")
print(f"\nveh_type distribution:\n{df['veh_type'].value_counts()}")
print(f"\npriority distribution:\n{df['priority'].value_counts()}")

event_cause filled from NLP: 2904 rows

event_cause distribution:
event_cause
vehicle_breakdown       4891
others                   619
pot_holes                512
water_logging            451
construction             426
accident                 365
tree_fall                283
road_conditions          156
congestion               136
public_event              76
procession                67
vip_movement              20
protest                   15
Debris                    10
test_demo                  3
Fog / Low Visibility       2
debris                     1
Name: count, dtype: int64

veh_type distribution:
veh_type
unknown          2896
bmtc_bus         1465
heavy_vehicle    1028
lcv               680
private_bus       460
others            449
private_car       345
truck             276
ksrtc_bus         217
taxi               95
car                61
auto               53
two_wheeler         8
Name: count, dtype: int64

priority distribution:
priority
High    4963
Low     3070

In [138]:
# ═══════════════════════════════════════════════════════════════
# CELL: Prepare final dataset — re-encode after NLP fill
# ═══════════════════════════════════════════════════════════════

from sklearn.preprocessing import LabelEncoder

# Re-encode all categoricals now that NLP has filled the gaps
cat_cols = ['event_cause', 'veh_type', 'corridor', 'police_station', 'zone']
le = {}
for col in cat_cols:
    df[col] = df[col].fillna('unknown')
    le[col] = LabelEncoder()
    df[col + '_enc'] = le[col].fit_transform(df[col])

# Resource tier target
def resource_tier(row):
    score = 0
    if row['priority'] == 'High':                                    score += 1
    if row['requires_road_closure'] == 1:                            score += 1
    if row['event_cause'] in ['tree_fall', 'accident', 'flooding']:  score += 1
    return min(score, 2)

df['resource_tier'] = df.apply(resource_tier, axis=1)

# Split normal vs extended
df_normal   = df[df['duration_min'] <= 720].copy()
df_extended = df[df['duration_min'] >  720].copy()

# dropna on STRUCT_FEATURES only — emb_* columns don't exist yet (added in transformer cell)
df_normal = df_normal.dropna(subset=STRUCT_FEATURES)

print(f"Normal incidents   : {len(df_normal)}")
print(f"Extended incidents : {len(df_extended)}")
print(f"\nResource tier distribution (normal):")
print(df_normal['resource_tier'].value_counts().sort_index())
print("\nTier 0 = 1 officer | Tier 1 = 2 officers | Tier 2 = 4 officers")

Normal incidents   : 7096
Extended incidents : 937

Resource tier distribution (normal):
resource_tier
0    2144
1    4460
2     492
Name: count, dtype: int64

Tier 0 = 1 officer | Tier 1 = 2 officers | Tier 2 = 4 officers


In [139]:
pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [140]:
# ═══════════════════════════════════════════════════════════════
# CELL: Transformer Embeddings
# Replaces TF-IDF+SVD with paraphrase-multilingual-MiniLM-L12-v2
# Handles Kannada natively — no translation needed
# ═══════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer
import numpy as np
import joblib

print("Loading multilingual transformer model...")
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded. Generating embeddings for all descriptions...")

# Use raw description directly — model handles Kannada natively
raw_desc = df['description'].fillna('').astype(str).tolist()
embeddings = embedder.encode(raw_desc, batch_size=64, show_progress_bar=True)
# Shape: (8033, 384)

EMB_DIM = embeddings.shape[1]  # 384
emb_df  = pd.DataFrame(
    embeddings,
    columns=[f'emb_{i}' for i in range(EMB_DIM)],
    index=df.index
)

# Drop old TF-IDF topics, add transformer embeddings
old_topic_cols = [c for c in df.columns if c.startswith('desc_topic_')]
df = df.drop(columns=old_topic_cols)
df = pd.concat([df, emb_df], axis=1)

# Update feature lists
TRANSFORMER_FEATURES = [f'emb_{i}' for i in range(EMB_DIM)]
ALL_FEATURES = STRUCT_FEATURES + TRANSFORMER_FEATURES

# Save embedder for inference
joblib.dump(embedder, r"C:\Users\bhard\Desktop\GRIDATHON\backend\app\models\sentence_embedder.pkl")

print(f"\nEmbeddings shape   : {embeddings.shape}")
print(f"Total features now : {len(ALL_FEATURES)}  ({len(STRUCT_FEATURES)} struct + {EMB_DIM} transformer)")
print(f"df shape           : {df.shape}")

Loading multilingual transformer model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3078.26it/s]


Model loaded. Generating embeddings for all descriptions...


Batches: 100%|██████████| 126/126 [00:37<00:00,  3.35it/s]



Embeddings shape   : (8033, 384)
Total features now : 397  (13 struct + 384 transformer)
df shape           : (8033, 449)


In [141]:
# ═══════════════════════════════════════════════════════════════
# CELL: Retrain with transformer features + fix resource model
# Resource target → priority (real label, not circular)
# ═══════════════════════════════════════════════════════════════

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, classification_report
import joblib, warnings
warnings.filterwarnings('ignore')

# ── Rebuild splits with transformer features ──────────────────
df_normal = df[df['duration_min'] <= 720].copy()
df_normal = df_normal.dropna(subset=ALL_FEATURES)

# Target 1: duration (regression)
# Target 2: priority — real label, not derived from features → no leakage
df_normal['priority_enc'] = (df_normal['priority'] == 'High').astype(int)

X         = df_normal[ALL_FEATURES]
y_dur     = df_normal['duration_min']
y_priority= df_normal['priority_enc']

X_train, X_test, y_dur_train, y_dur_test, y_pri_train, y_pri_test = train_test_split(
    X, y_dur, y_priority, test_size=0.2, random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Duration model (XGBoost — already best) ───────────────────
print("=" * 58)
print("  DURATION MODEL (transformer features)")
print("=" * 58)
dur_model = XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
cv_dur = cross_val_score(dur_model, X_train, y_dur_train,
                          cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"  CV MAE  : {-cv_dur.mean():.1f} min")
dur_model.fit(X_train, y_dur_train)
print(f"  Test MAE: {mean_absolute_error(y_dur_test, dur_model.predict(X_test)):.1f} min")
print(f"  (Previous XGBoost with TF-IDF: 50.8 min)")

# ── Priority / resource model (fixed — no leakage) ────────────
print("\n" + "=" * 58)
print("  PRIORITY MODEL (real label — no circular dependency)")
print("=" * 58)

pri_models = {
    'XGBoost   [Boosting]': XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'),
    'GradBoost [Boosting]': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    'RandomForest[Bagging]': RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight='balanced',
        random_state=42, n_jobs=-1),
}

pri_results = {}
for name, model in pri_models.items():
    cv = cross_val_score(model, X_train, y_pri_train,
                         cv=kf, scoring='f1_weighted', n_jobs=-1)
    pri_results[name] = cv.mean()
    print(f"  {name:<25}  CV F1 = {pri_results[name]:.3f}")

best_pri_name  = max(pri_results, key=pri_results.get)
pri_model      = pri_models[best_pri_name]
pri_model.fit(X_train, y_pri_train)
print(f"\n  Best: {best_pri_name.strip()}")
print(classification_report(y_pri_test, pri_model.predict(X_test),
      target_names=['Low Priority', 'High Priority']))

# ── Personnel count from priority + duration + road_closure ───
def get_personnel_v2(priority_pred, duration_min, requires_road_closure):
    """
    priority_pred        : 0=Low, 1=High
    duration_min         : predicted duration
    requires_road_closure: 0 or 1
    """
    base = 1
    if priority_pred == 1:         base += 1   # High priority → +1
    if requires_road_closure:      base += 1   # Road blocked  → +1
    if duration_min > 240:         base += 1   # Relief shift  → +1
    if duration_min > 480:         base += 1   # 2nd shift     → +1
    return base

def get_urgency_v2(priority_pred, duration_min, requires_road_closure):
    if requires_road_closure and priority_pred == 1: return 'CRITICAL'
    if requires_road_closure or priority_pred == 1:  return 'HIGH'
    if duration_min > 180:                           return 'MODERATE'
    return 'NORMAL'

# ── Save updated models ───────────────────────────────────────
joblib.dump(dur_model,  r"C:\Users\bhard\Desktop\GRIDATHON\backend\app\models\duration_model.pkl")
joblib.dump(pri_model,  r"C:\Users\bhard\Desktop\GRIDATHON\backend\app\models\resource_model.pkl")
joblib.dump(le,         r"C:\Users\bhard\Desktop\GRIDATHON\backend\app\models\label_encoders.pkl")
print("\nUpdated models saved.")

  DURATION MODEL (transformer features)
  CV MAE  : 54.2 min
  Test MAE: 50.6 min
  (Previous XGBoost with TF-IDF: 50.8 min)

  PRIORITY MODEL (real label — no circular dependency)
  XGBoost   [Boosting]       CV F1 = 0.998
  GradBoost [Boosting]       CV F1 = 0.998
  RandomForest[Bagging]      CV F1 = 0.886

  Best: XGBoost   [Boosting]
               precision    recall  f1-score   support

 Low Priority       1.00      1.00      1.00       540
High Priority       1.00      1.00      1.00       880

     accuracy                           1.00      1420
    macro avg       1.00      1.00      1.00      1420
 weighted avg       1.00      1.00      1.00      1420


Updated models saved.


In [143]:
# ═══════════════════════════════════════════════════════════════
# CELL: Final Inference Function (transformer + fixed models)
# ═══════════════════════════════════════════════════════════════

def predict_deployment_v2(
    description: str,
    latitude: float,
    longitude: float,
    requires_road_closure: bool = False,
    event_cause: str    = 'others',
    veh_type: str       = 'unknown',
    corridor: str       = 'Non-corridor',
    police_station: str = 'unknown',
    zone: str           = 'unknown',
    hour: int           = None,
    day_of_week: int    = None,
    month: int          = None,
) -> dict:
    import datetime
    now = datetime.datetime.now()
    if hour        is None: hour        = now.hour
    if day_of_week is None: day_of_week = now.weekday()
    if month       is None: month       = now.month

    # ── Transformer embedding directly on raw text ────────────
    emb = embedder.encode([description])[0]

    # ── NLP keyword fill for structured fields ────────────────
    desc_lower = description.lower()
    if event_cause == 'others':
        nlp = extract_event_cause(desc_lower)
        if nlp: event_cause = nlp
    if veh_type == 'unknown':
        nlp = extract_veh_type(desc_lower)
        if nlp: veh_type = nlp

    # ── Encode categoricals safely ────────────────────────────
    def safe_encode(encoder, value):
        """Try value → first known class → index 0."""
        known = set(encoder.classes_)
        if value in known:
            return int(encoder.transform([value])[0])
        # pick first known class that was seen during training
        return int(encoder.transform([encoder.classes_[0]])[0])

    struct_row = {
        'hour'                 : hour,
        'day_of_week'          : day_of_week,
        'month'                : month,
        'is_night'             : 1 if (hour >= 21 or hour <= 6) else 0,
        'reporting_delay_min'  : 0,
        'latitude'             : latitude,
        'longitude'            : longitude,
        'requires_road_closure': int(requires_road_closure),
        'event_cause_enc'      : safe_encode(le['event_cause'],    event_cause),
        'veh_type_enc'         : safe_encode(le['veh_type'],       veh_type),
        'corridor_enc'         : safe_encode(le['corridor'],        corridor),
        'police_station_enc'   : safe_encode(le['police_station'], police_station),
        'zone_enc'             : safe_encode(le['zone'],            zone),
    }
    emb_row = {f'emb_{i}': float(emb[i]) for i in range(len(emb))}
    row     = {**struct_row, **emb_row}

    X_input = pd.DataFrame([row])[ALL_FEATURES]

    # ── Predict duration + priority ───────────────────────────
    duration_min  = float(dur_model.predict(X_input)[0])
    priority_pred = int(pri_model.predict(X_input)[0])
    personnel     = get_personnel_v2(priority_pred, duration_min, requires_road_closure)
    urgency       = get_urgency_v2(priority_pred, duration_min, requires_road_closure)

    return {
        'estimated_duration_min' : round(duration_min, 1),
        'estimated_duration_hrs' : round(duration_min / 60, 2),
        'priority'               : 'High' if priority_pred == 1 else 'Low',
        'personnel_to_deploy'    : personnel,
        'urgency'                : urgency,
        'detected_cause'         : event_cause,
        'detected_veh_type'      : veh_type,
    }

# ── Test 3 scenarios ──────────────────────────────────────────
scenarios = [
    dict(description="Heavy truck engine stalled blocking entire lane on highway",
         latitude=13.040004, longitude=77.518099,
         requires_road_closure=True, corridor='Tumkur Road', police_station='Peenya'),

    dict(description="ಮರ ಬಿದ್ದಿದೆ ರಸ್ತೆ ಬ್ಲಾಕ್ ಆಗಿದೆ",
         latitude=12.955622, longitude=77.585708,
         requires_road_closure=True),

    dict(description="Minor vehicle breakdown, moving to side",
         latitude=12.921876, longitude=77.645158,
         requires_road_closure=False, corridor='ORR East 1'),
]

for i, sc in enumerate(scenarios, 1):
    result = predict_deployment_v2(**sc)
    print(f"\n{'='*50}")
    print(f"  Scenario {i}: {sc['description'][:45]}")
    print(f"{'='*50}")
    for k, v in result.items():
        print(f"  {k:<28}: {v}")


  Scenario 1: Heavy truck engine stalled blocking entire la
  estimated_duration_min      : 111.8
  estimated_duration_hrs      : 1.86
  priority                    : High
  personnel_to_deploy         : 3
  urgency                     : CRITICAL
  detected_cause              : vehicle_breakdown
  detected_veh_type           : heavy_vehicle

  Scenario 2: ಮರ ಬಿದ್ದಿದೆ ರಸ್ತೆ ಬ್ಲಾಕ್ ಆಗಿದೆ
  estimated_duration_min      : 65.7
  estimated_duration_hrs      : 1.1
  priority                    : Low
  personnel_to_deploy         : 2
  urgency                     : HIGH
  detected_cause              : others
  detected_veh_type           : unknown

  Scenario 3: Minor vehicle breakdown, moving to side
  estimated_duration_min      : 126.1
  estimated_duration_hrs      : 2.1
  priority                    : High
  personnel_to_deploy         : 2
  urgency                     : HIGH
  detected_cause              : vehicle_breakdown
  detected_veh_type           : unknown


In [144]:
pip install anthropic

Defaulting to user installation because normal site-packages is not writeable
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/929.8 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/929.8 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 929.8/929.8 kB 3.0 MB/s eta 0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 4.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.5 MB/s eta 0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ----- ---


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [145]:
# ═══════════════════════════════════════════════════════════════
# CELL: LLM Firewall + Worker Router
# Stage 1 — Claude Haiku validates the input description
# Stage 2 — Prediction worker runs only if input is valid
# ═══════════════════════════════════════════════════════════════

import anthropic
import json
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError

# ── Semantic similarity fallback (no API key needed) ──────────
# Average embeddings of known incident types as reference anchors
_ANCHOR_TEXTS = [
    "vehicle breakdown on road",
    "tree fell on road blocking traffic",
    "accident collision crash injured",
    "road flooding waterlogging",
    "pothole road damage crater",
    "traffic congestion blocked lane",
    "bus stalled engine problem",
    "truck breakdown highway",
]
_anchor_embeddings = embedder.encode(_ANCHOR_TEXTS)

def _semantic_similarity_check(description: str, threshold: float = 0.25) -> tuple:
    """Fast local check — cosine similarity to known incident anchor texts."""
    emb = embedder.encode([description])[0]
    emb_norm     = emb / (np.linalg.norm(emb) + 1e-8)
    anchor_norms = _anchor_embeddings / (np.linalg.norm(_anchor_embeddings, axis=1, keepdims=True) + 1e-8)
    sims         = anchor_norms @ emb_norm
    max_sim      = float(sims.max())
    best_anchor  = _ANCHOR_TEXTS[int(sims.argmax())]
    return max_sim >= threshold, max_sim, best_anchor

# ── LLM Firewall (Claude Haiku) ───────────────────────────────
def llm_firewall(description: str) -> tuple:
    """
    Returns (is_valid: bool, reason: str, detected_type: str)
    Uses Claude Haiku for speed + cost efficiency.
    Falls back to semantic similarity if API key not set.
    """
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")

    # Fast semantic pre-check — avoids API call for clearly valid inputs
    sim_valid, sim_score, best_match = _semantic_similarity_check(description)
    if sim_valid and sim_score > 0.55:
        return True, f"Semantic match ({sim_score:.2f}) → {best_match}", "traffic_incident"

    # Clearly nonsense (very low similarity) → reject immediately
    if sim_score < 0.08:
        return False, f"Input does not resemble any known incident type (similarity={sim_score:.2f})", None

    # Uncertain range (0.08–0.55) → call Claude for final judgment
    if not api_key:
        # No API key: trust semantic score
        return sim_valid, f"Semantic score: {sim_score:.2f}", "uncertain"

    try:
        client = anthropic.Anthropic(api_key=api_key)
        resp   = client.messages.create(
            model      = "claude-haiku-4-5-20251001",
            max_tokens = 120,
            messages   = [{
                "role": "user",
                "content": (
                    "You are a traffic incident management system validator.\n"
                    "Decide if this text describes a genuine road/traffic incident "
                    "(breakdown, accident, tree fall, flooding, congestion, road damage, etc.).\n\n"
                    f'Text: "{description}"\n\n'
                    'Reply ONLY with valid JSON: '
                    '{"valid": true/false, "reason": "...", "incident_type": "..."}'
                )
            }]
        )
        data = json.loads(resp.content[0].text.strip())
        return bool(data["valid"]), data.get("reason", ""), data.get("incident_type", "unknown")
    except Exception as e:
        # API failed → fall back to semantic result
        return sim_valid, f"API error ({e}), using semantic fallback: {sim_score:.2f}", "fallback"

# ── Worker Router ─────────────────────────────────────────────
def incident_pipeline(description: str, **kwargs) -> dict:
    """
    Two-stage worker pipeline:
      Worker 1 → LLM firewall (validates input)
      Worker 2 → Prediction (runs only if valid)
    """
    with ThreadPoolExecutor(max_workers=2) as pool:

        # ── Stage 1: Firewall worker ──────────────────────────
        fw_future = pool.submit(llm_firewall, description)
        try:
            is_valid, reason, incident_type = fw_future.result(timeout=15)
        except FutureTimeoutError:
            is_valid, reason, incident_type = False, "Firewall timed out", None

        if not is_valid:
            return {
                "status"              : "REJECTED",
                "reason"              : reason,
                "estimated_duration_min": None,
                "personnel_to_deploy" : None,
                "urgency"             : None,
            }

        # ── Stage 2: Prediction worker ────────────────────────
        pred_future = pool.submit(predict_deployment_v2, description, **kwargs)
        try:
            result = pred_future.result(timeout=30)
        except FutureTimeoutError:
            return {"status": "ERROR", "reason": "Prediction timed out"}

        result["status"]         = "PROCESSED"
        result["firewall_note"]  = reason
        result["incident_type"]  = incident_type
        return result


# ── Test scenarios ────────────────────────────────────────────
test_cases = [
    # Valid incidents
    dict(description="Heavy truck broke down blocking the left lane on Tumkur Road",
         latitude=13.040004, longitude=77.518099,
         requires_road_closure=True, corridor='Tumkur Road'),

    dict(description="ಮರ ಬಿದ್ದಿದೆ ರಸ್ತೆ ತಡೆಯಾಗಿದೆ",   # Kannada: tree fell, road blocked
         latitude=12.955622, longitude=77.585708,
         requires_road_closure=True),

    # Absurd / irrelevant inputs — should be rejected
    dict(description="I love pizza and watching netflix",
         latitude=12.921876, longitude=77.645158),

    dict(description="asdfghjkl qwerty random gibberish 12345",
         latitude=12.921876, longitude=77.645158),
]

for i, tc in enumerate(test_cases, 1):
    print(f"\n{'═'*52}")
    print(f"  Test {i}: \"{tc['description'][:48]}\"")
    print(f"{'═'*52}")
    result = incident_pipeline(**tc)
    for k, v in result.items():
        print(f"  {k:<28}: {v}")


════════════════════════════════════════════════════
  Test 1: "Heavy truck broke down blocking the left lane on"
════════════════════════════════════════════════════
  estimated_duration_min      : 147.1
  estimated_duration_hrs      : 2.45
  priority                    : High
  personnel_to_deploy         : 3
  urgency                     : CRITICAL
  detected_cause              : vehicle_breakdown
  detected_veh_type           : heavy_vehicle
  status                      : PROCESSED
  firewall_note               : Semantic match (0.79) → truck breakdown highway
  incident_type               : traffic_incident

════════════════════════════════════════════════════
  Test 2: "ಮರ ಬಿದ್ದಿದೆ ರಸ್ತೆ ತಡೆಯಾಗಿದೆ"
════════════════════════════════════════════════════
  estimated_duration_min      : 56.9
  estimated_duration_hrs      : 0.95
  priority                    : Low
  personnel_to_deploy         : 2
  urgency                     : HIGH
  detected_cause              : others
  detected_